
# InvariantRRF — Information Fusion final validation

This notebook is a **targeted follow-up** to the Information Fusion strengthening audit.

It does **not** add more datasets or retrain retrieval models. It answers the two remaining methodological questions that matter before rewriting the paper:

1. **Does plain nested/hierarchical fusion remain replication-sensitive across a broader additive rank-fusion class?**  
   The previous strengthening run showed that nested fusion can be an excellent empirical mitigation for highly redundant families. Here we test the harder property: whether duplicating one member of an already multi-member real family can change nested fusion.

2. **Does the Stable-style prefix certificate match strict completion stability under a systematically enumerated open-tail model?**  
   The previous run exhaustively completed 240 randomly sampled small cases. Here we enumerate a deterministic finite state space of prefix configurations, weight combinations, kernels, and top-\(K\) values.

The notebook uses the same frozen canonical and SPLADE closure archives. No external model downloads are required.

## Required Kaggle inputs

Attach:

- `InvariantRRF_Canonical_STRICT_DeepDense_TaskAdaptiveK_Results.zip`
- `InvariantRRF_V43_SPLADE_TwoCheckpoint_Closure.zip`
- the benchmark bundle containing `mpdr/data/*_mpdr/dev/qrels.tsv`

## Output

The last cell writes:

`InvariantRRF_InformationFusion_Final_Validation_Results.zip`


In [1]:

from pathlib import Path
from collections import defaultdict
from itertools import permutations, product
import hashlib, json, math, shutil, zipfile

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import wilcoxon

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
OUT = ROOT / 'InvariantRRF_InformationFusion_Final_Validation'
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
(OUT/'tables').mkdir()
(OUT/'statistics').mkdir()

TOP_K = 10
DEPTHS = [50, 100]
COPY_EXTRAS = [1, 3, 7]  # original member + 1/3/7 copies => multiplicity 2/4/8
RBO_P = 0.90
SEED = 20260909

KERNELS = ['rrf60','inverse_rank','inverse_sqrt','exp20','log_discount','borda']

CANONICAL_ZIP = None
SPLADE_CLOSURE_ZIP = None

print('INPUT_ROOT:', INPUT_ROOT)
print('OUT:', OUT)


INPUT_ROOT: /kaggle/input
OUT: /kaggle/working/InvariantRRF_InformationFusion_Final_Validation


## 1. Discover and verify frozen inputs

In [2]:

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def discover_zip(keywords):
    scored = []
    for p in INPUT_ROOT.rglob('*.zip'):
        n = p.name.lower()
        score = sum(k.lower() in n for k in keywords)
        if score:
            scored.append((score, len(n), str(p), p))
    if not scored:
        return None
    scored.sort(reverse=True)
    return scored[0][-1]

def find_canonical_root(root):
    root = Path(root)
    if (root/'runs').is_dir():
        return root
    for m in sorted(root.rglob('RUN_MANIFEST.json')):
        if (m.parent/'runs').is_dir():
            return m.parent
    for d in sorted(p for p in root.rglob('runs') if p.is_dir()):
        if any(d.glob('*_dense_STRICT_top1000.json')):
            return d.parent
    raise FileNotFoundError(f'No canonical root under {root}')

def find_splade_root(root):
    root = Path(root)
    candidates = []
    if (root/'generated_runs').is_dir():
        candidates.append(root)
    candidates += [p.parent for p in root.rglob('SPLADE_V43_CLOSURE_PROTOCOL.json')]
    candidates += [p.parent for p in root.rglob('CLOSURE_MANIFEST.json')
                   if (p.parent/'generated_runs').is_dir()]
    for c in candidates:
        if (c/'generated_runs'/'SciFact_splade_selfdistil_top500.json').exists():
            return c
    return None

cz = Path(CANONICAL_ZIP) if CANONICAL_ZIP else discover_zip(
    ['invariantrrf','canonical','deepdense','taskadaptivek']
)
if cz is None:
    raise FileNotFoundError('Canonical result ZIP not found')
cex = ROOT/'_if_final_canonical'
if cex.exists():
    shutil.rmtree(cex)
cex.mkdir()
with zipfile.ZipFile(cz,'r') as zf:
    zf.extractall(cex)
CANON = find_canonical_root(cex)
RUNS = CANON/'runs'

manifest_path = CANON/'RUN_MANIFEST.json'
manifest_status = {'present': manifest_path.exists(), 'checked': 0, 'failures': []}
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text('utf-8'))
    for rel, expected in manifest.get('files',{}).items():
        p = CANON/rel
        if not p.exists():
            manifest_status['failures'].append((rel,'MISSING'))
        elif sha256_file(p) != expected:
            manifest_status['failures'].append((rel,'HASH_MISMATCH'))
        manifest_status['checked'] += 1
    if manifest_status['failures']:
        raise AssertionError(manifest_status['failures'][:10])

sz = Path(SPLADE_CLOSURE_ZIP) if SPLADE_CLOSURE_ZIP else discover_zip(
    ['splade','twocheckpoint','closure']
)
SPLADE = None
if sz is not None:
    sex = ROOT/'_if_final_splade'
    if sex.exists():
        shutil.rmtree(sex)
    sex.mkdir()
    with zipfile.ZipFile(sz,'r') as zf:
        zf.extractall(sex)
    SPLADE = find_splade_root(sex)

SELF_RUNS = SPLADE/'generated_runs' if SPLADE else None

print('Canonical ZIP:', cz)
print('Canonical manifest:', 'PASS' if not manifest_status['failures'] else 'FAIL',
      manifest_status['checked'], 'files')
print('SPLADE ZIP:', sz if sz else 'NOT FOUND')


Canonical ZIP: /kaggle/input/notebooks/asmaebr/invariantrrf-canonical-strict-fourdataset-rerun/InvariantRRF_Canonical_STRICT_DeepDense_TaskAdaptiveK_Results.zip
Canonical manifest: PASS 62 files
SPLADE ZIP: /kaggle/input/notebooks/asmaebr/invariantrrf-v43-splade-twocheckpoint-closure-audi/InvariantRRF_V43_SPLADE_TwoCheckpoint_Closure.zip


## 2. Runs, qrels, and fusion helpers

In [3]:

def load_run(path):
    raw = json.loads(Path(path).read_text('utf-8'))
    out = {}
    for qid, seq in raw.items():
        norm = []
        for x in seq:
            if isinstance(x, (list, tuple)):
                norm.append((str(x[0]), float(x[1]) if len(x) > 1 else 0.0))
            else:
                norm.append((str(x), 0.0))
        ids = [d for d,_ in norm]
        if len(ids) != len(set(ids)):
            raise AssertionError(f'Duplicate document IDs in {path}/{qid}')
        out[str(qid)] = norm
    return out

def run_path(ds, source):
    mapping = {
        'bm25': RUNS/f'{ds}_bm25_top1000.json',
        'bm25_lowb': RUNS/f'{ds}_bm25_lowb_top1000.json',
        'bm25_highb': RUNS/f'{ds}_bm25_highb_top1000.json',
        'dense': RUNS/f'{ds}_dense_STRICT_top1000.json',
        'splade_ensemble': RUNS/f'{ds}_splade_ensemble_top500.json',
    }
    if source == 'splade_self':
        if SELF_RUNS is None:
            raise FileNotFoundError('SPLADE closure unavailable')
        return SELF_RUNS/f'{ds}_splade_selfdistil_top500.json'
    return mapping[source]

RUN_CACHE = {}
def get_run(ds, source):
    key = (ds,source)
    if key not in RUN_CACHE:
        RUN_CACHE[key] = load_run(run_path(ds,source))
    return RUN_CACHE[key]

DATASET_DIRNAMES = {
    'SciFact':'scifact_mpdr',
    'TREC-COVID':'trec-covid_mpdr',
    'FiQA':'fiqa_mpdr',
    'ArguAna':'arguana_mpdr',
}

def find_dataset_root(dirname):
    for p in INPUT_ROOT.rglob(dirname):
        if p.is_dir() and (p/'dev'/'qrels.tsv').exists():
            return p
    return None

def load_qrels(root):
    if root is None:
        return {}
    out = defaultdict(dict)
    for line in (Path(root)/'dev'/'qrels.tsv').read_text('utf-8').splitlines():
        sp = line.split('\t')
        if len(sp) < 2:
            continue
        qid,did = str(sp[0]),str(sp[1])
        try:
            rel = float(sp[2]) if len(sp) >= 3 and sp[2] else 1.0
        except Exception:
            rel = 1.0
        if rel > 0:
            out[qid][did] = rel
    return dict(out)

ROOTS = {ds: find_dataset_root(dn) for ds,dn in DATASET_DIRNAMES.items()}
QRELS = {ds: load_qrels(ROOTS[ds]) for ds in ROOTS}

def prefix_docs(seq, depth):
    return [str(d) for d,_ in seq[:min(depth,len(seq))]]

def top_docs(res, k=TOP_K):
    return [d for d,_ in res[:k]]

def ndcg_at_k(docids, rels, k=10):
    if not rels:
        return np.nan
    dcg = 0.0
    for i,d in enumerate(docids[:k], start=1):
        rel = float(rels.get(str(d),0.0))
        dcg += (2.0**rel - 1.0) / math.log2(i+1.0)
    ideal = sorted((float(v) for v in rels.values()), reverse=True)[:k]
    if not ideal:
        return 0.0
    idcg = sum((2.0**rel - 1.0)/math.log2(i+2.0) for i,rel in enumerate(ideal))
    return dcg/idcg if idcg > 0 else 0.0

def kernel_value(name, r, cap):
    r = float(r)
    cap = max(1,int(cap))
    if name == 'rrf60':
        return 61.0/(60.0+r)
    if name == 'inverse_rank':
        return 1.0/r
    if name == 'inverse_sqrt':
        return 1.0/math.sqrt(r)
    if name == 'exp20':
        return math.exp(-(r-1.0)/20.0)
    if name == 'log_discount':
        return 1.0/math.log2(r+1.0)
    if name == 'borda':
        return max(0.0,(cap-r+1.0)/cap)
    raise KeyError(name)

def additive_fuse(rankings, depth, kernel, weights=None):
    weights = weights or {n:1.0 for n in rankings}
    scores = defaultdict(float)
    for name,seq in rankings.items():
        w = float(weights.get(name,1.0))
        for r,did in enumerate(prefix_docs(seq,depth), start=1):
            scores[did] += w*kernel_value(kernel,r,depth)
    return sorted(scores.items(), key=lambda x:(-x[1],x[0]))

def signature(seq, depth):
    ids = prefix_docs(seq,depth)
    return hashlib.sha256('\x1f'.join(ids).encode()).hexdigest()

def unique_family_representatives(rankings, family_map, depth):
    fams = defaultdict(list)
    for name in rankings:
        fams[str(family_map[name])].append(name)
    reps = {}
    rep_family = {}
    for fam,members in fams.items():
        bysig = defaultdict(list)
        for m in members:
            bysig[signature(rankings[m],depth)].append(m)
        for same in bysig.values():
            rep = sorted(same)[0]
            reps[rep] = rankings[rep]
            rep_family[rep] = fam
    return reps,rep_family

def family_budget_fuse(rankings, family_map, depth, kernel):
    reps,rep_family = unique_family_representatives(rankings,family_map,depth)
    fam_members = defaultdict(list)
    for rep,fam in rep_family.items():
        fam_members[fam].append(rep)
    weights = {}
    for fam,members in fam_members.items():
        for m in members:
            weights[m] = 1.0/len(members)
    return additive_fuse(reps,depth,kernel,weights)

def nested_family_fuse(family_rankings, outside_rankings, depth, kernel):
    inner = additive_fuse(family_rankings,depth,kernel)
    inner_seq = [(d,s) for d,s in inner[:depth]]
    return additive_fuse({'FAMILY_CONSENSUS':inner_seq, **outside_rankings},depth,kernel)

def holm_adjust(pvals):
    p = np.asarray(pvals,dtype=float)
    m = len(p)
    if m == 0:
        return p
    order = np.argsort(p)
    out = np.empty(m,float)
    running = 0.0
    for j,idx in enumerate(order):
        val = (m-j)*p[idx]
        running = max(running,val)
        out[idx] = min(1.0,running)
    return out

def wilcoxon_safe(x):
    x = np.asarray(x,float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    if np.allclose(x,0):
        return 1.0
    try:
        return float(wilcoxon(x,zero_method='wilcox',alternative='two-sided').pvalue)
    except Exception:
        return np.nan

print({ds: len(QRELS[ds]) for ds in ['SciFact','TREC-COVID','FiQA','ArguAna']})


{'SciFact': 300, 'TREC-COVID': 50, 'FiQA': 648, 'ArguAna': 1401}



## 3. Cross-kernel replication audit of nested fusion on real multi-member families

The previous strengthening audit showed that nested fusion can preserve a clean parent almost perfectly when an entire synthetic family consists of tiny perturbations of that parent. That is a useful empirical property, not a formal replication guarantee.

This test starts from an **already multi-member real family** and then duplicates one member. For each attacked member, we compare the post-copy result with the method's own pre-copy result.

Real family conditions:

- BM25 parameter family on SciFact, FiQA, ArguAna: 3 attackable members each.
- SPLADE checkpoint family on SciFact and ArguAna: 2 attackable members each.

That gives 13 attacked-member conditions per kernel/depth/copy level, matching the original RRF nested audit structure.


In [4]:

REAL_SCENARIOS = []
for ds in ['SciFact','FiQA','ArguAna']:
    outside = ['dense']
    if (RUNS/f'{ds}_splade_ensemble_top500.json').exists():
        outside = ['splade_ensemble','dense']
    REAL_SCENARIOS.append({
        'scenario':'BM25-family',
        'dataset':ds,
        'family_sources':['bm25','bm25_lowb','bm25_highb'],
        'outside_sources':outside,
    })

if SELF_RUNS is not None:
    for ds in ['SciFact','ArguAna']:
        if (SELF_RUNS/f'{ds}_splade_selfdistil_top500.json').exists():
            REAL_SCENARIOS.append({
                'scenario':'SPLADE-family',
                'dataset':ds,
                'family_sources':['splade_ensemble','splade_self'],
                'outside_sources':['bm25'],
            })

print([(x['scenario'],x['dataset'],x['family_sources']) for x in REAL_SCENARIOS])

nested_perq = []

for sc in REAL_SCENARIOS:
    ds = sc['dataset']
    fam_sources = sc['family_sources']
    outside_sources = sc['outside_sources']
    runs = {s:get_run(ds,s) for s in set(fam_sources+outside_sources)}
    qids = sorted(set.intersection(*(set(runs[s]) for s in runs)))

    for depth in DEPTHS:
        for kernel in KERNELS:
            for attacked in fam_sources:
                for extra_copies in COPY_EXTRAS:
                    for q in qids:
                        family = {s:runs[s][q] for s in fam_sources}
                        outside = {s:runs[s][q] for s in outside_sources}
                        all_base = {**family,**outside}

                        fmap_base = {s:'FAMILY' for s in fam_sources}
                        fmap_base.update({s:f'OUT::{s}' for s in outside_sources})

                        base_flat = top_docs(additive_fuse(all_base,depth,kernel))
                        base_nested = top_docs(nested_family_fuse(family,outside,depth,kernel))
                        base_fb = top_docs(family_budget_fuse(all_base,fmap_base,depth,kernel))

                        family_attacked = dict(family)
                        all_attacked = dict(all_base)
                        fmap_attacked = dict(fmap_base)
                        for j in range(extra_copies):
                            name = f'{attacked}__copy{j+1}'
                            family_attacked[name] = family[attacked]
                            all_attacked[name] = family[attacked]
                            fmap_attacked[name] = 'FAMILY'

                        got_flat = top_docs(additive_fuse(all_attacked,depth,kernel))
                        got_nested = top_docs(nested_family_fuse(family_attacked,outside,depth,kernel))
                        got_fb = top_docs(family_budget_fuse(all_attacked,fmap_attacked,depth,kernel))

                        rels = QRELS[ds].get(q,{})
                        bfn = ndcg_at_k(base_flat,rels,TOP_K)
                        gfn = ndcg_at_k(got_flat,rels,TOP_K)
                        bnn = ndcg_at_k(base_nested,rels,TOP_K)
                        gnn = ndcg_at_k(got_nested,rels,TOP_K)
                        bmn = ndcg_at_k(base_fb,rels,TOP_K)
                        gmn = ndcg_at_k(got_fb,rels,TOP_K)

                        nested_perq.append({
                            'scenario':sc['scenario'],
                            'dataset':ds,
                            'depth':depth,
                            'kernel':kernel,
                            'attacked_member':attacked,
                            'extra_copies':extra_copies,
                            'total_attacked_multiplicity':1+extra_copies,
                            'qid':q,
                            'flat_order_preserved':int(got_flat==base_flat),
                            'flat_set_preserved':int(set(got_flat)==set(base_flat)),
                            'nested_order_preserved':int(got_nested==base_nested),
                            'nested_set_preserved':int(set(got_nested)==set(base_nested)),
                            'family_budget_order_preserved':int(got_fb==base_fb),
                            'family_budget_set_preserved':int(set(got_fb)==set(base_fb)),
                            'flat_abs_ndcg_drift':abs(gfn-bfn) if np.isfinite(bfn) else np.nan,
                            'nested_abs_ndcg_drift':abs(gnn-bnn) if np.isfinite(bnn) else np.nan,
                            'family_budget_abs_ndcg_drift':abs(gmn-bmn) if np.isfinite(bmn) else np.nan,
                        })

nested_perq_df = pd.DataFrame(nested_perq)
nested_perq_df.to_csv(OUT/'tables'/'nested_replication_per_query.csv',index=False)

group_cols = ['scenario','dataset','depth','kernel','attacked_member','extra_copies']
nested_summary = nested_perq_df.groupby(group_cols,as_index=False).agg(
    n_queries=('qid','count'),
    flat_order=('flat_order_preserved','mean'),
    flat_set=('flat_set_preserved','mean'),
    nested_order=('nested_order_preserved','mean'),
    nested_set=('nested_set_preserved','mean'),
    family_budget_order=('family_budget_order_preserved','mean'),
    family_budget_set=('family_budget_set_preserved','mean'),
    flat_abs_drift=('flat_abs_ndcg_drift','mean'),
    nested_abs_drift=('nested_abs_ndcg_drift','mean'),
    family_budget_abs_drift=('family_budget_abs_ndcg_drift','mean'),
)

# Per-condition significance for nested post-copy nDCG drift vs exact zero is not meaningful.
# Instead test whether nested drift is systematically larger than family-budget drift query-wise.
pvals = []
for keys,g in nested_perq_df.groupby(group_cols,sort=False):
    diff = g['nested_abs_ndcg_drift'].to_numpy(float) - g['family_budget_abs_ndcg_drift'].to_numpy(float)
    pvals.append((*keys, wilcoxon_safe(diff)))
p_df = pd.DataFrame(pvals,columns=group_cols+['p_raw_nested_vs_family_budget_drift'])
p_df['p_holm_all'] = holm_adjust(p_df['p_raw_nested_vs_family_budget_drift'].fillna(1).to_numpy(float))
nested_summary = nested_summary.merge(p_df,on=group_cols,how='left')
nested_summary.to_csv(OUT/'tables'/'nested_replication_summary.csv',index=False)

assert (nested_summary['family_budget_order'] == 1.0).all()
assert (nested_summary['family_budget_set'] == 1.0).all()
assert np.allclose(nested_summary['family_budget_abs_drift'].fillna(0),0)

macro = nested_summary.groupby(['depth','kernel','extra_copies'],as_index=False).agg(
    conditions=('dataset','count'),
    flat_order=('flat_order','mean'),
    flat_set=('flat_set','mean'),
    nested_order=('nested_order','mean'),
    nested_set=('nested_set','mean'),
    family_budget_order=('family_budget_order','mean'),
    family_budget_set=('family_budget_set','mean'),
    flat_abs_drift=('flat_abs_drift','mean'),
    nested_abs_drift=('nested_abs_drift','mean'),
    family_budget_abs_drift=('family_budget_abs_drift','mean'),
    nested_significant_vs_fb=('p_holm_all',lambda x:int((x<.05).sum())),
)
macro.to_csv(OUT/'tables'/'nested_replication_macro.csv',index=False)
display(macro)
print('Family-budget exact-copy invariance: PASS in all',
      len(nested_summary), 'real-family/kernel/depth/copy conditions')


[('BM25-family', 'SciFact', ['bm25', 'bm25_lowb', 'bm25_highb']), ('BM25-family', 'FiQA', ['bm25', 'bm25_lowb', 'bm25_highb']), ('BM25-family', 'ArguAna', ['bm25', 'bm25_lowb', 'bm25_highb']), ('SPLADE-family', 'SciFact', ['splade_ensemble', 'splade_self']), ('SPLADE-family', 'ArguAna', ['splade_ensemble', 'splade_self'])]


,depth,kernel,extra_copies,conditions,flat_order,flat_set,nested_order,nested_set,family_budget_order,family_budget_set,flat_abs_drift,nested_abs_drift,family_budget_abs_drift,nested_significant_vs_fb
0,50,borda,1,13,0.084889,0.540591,0.374613,0.764970,1.0,1.0,0.013493,0.006660,0.0,10
1,50,borda,3,13,0.011180,0.271936,0.220594,0.628011,1.0,1.0,0.029533,0.011570,0.0,10
2,50,borda,7,13,0.003404,0.148567,0.171972,0.547611,1.0,1.0,0.045956,0.015043,0.0,12
3,50,exp20,1,13,0.079221,0.533467,0.357622,0.745305,1.0,1.0,0.014244,0.006305,0.0,10
4,50,exp20,3,13,0.009771,0.288702,0.206194,0.606200,1.0,1.0,0.032286,0.011580,0.0,10
5,50,exp20,7,13,0.003202,0.173243,0.165394,0.537875,1.0,1.0,0.049765,0.014362,0.0,11
6,50,inverse_rank,1,13,0.046045,0.477966,0.402694,0.749232,1.0,1.0,0.016103,0.008025,0.0,10
7,50,inverse_rank,3,13,0.004668,0.182447,0.240934,0.602381,1.0,1.0,0.034612,0.013322,0.0,10
8,50,inverse_rank,7,13,0.001684,0.101559,0.202785,0.532809,1.0,1.0,0.050024,0.014803,0.0,12
9,50,inverse_sqrt,1,13,0.059095,0.485761,0.383489,0.740714,1.0,1.0,0.018963,0.007768,0.0,10


Family-budget exact-copy invariance: PASS in all 468 real-family/kernel/depth/copy conditions



## 4. Systematic finite-state audit of the Stable-style certificate

The earlier notebook sampled 240 prefix states randomly and then exhaustively enumerated each state's completions.

Here we remove the random prefix-state sampling for a fixed small model:

- 4-document universe;
- 2 sources;
- observed prefix length 1 or 2 per source;
- source weights in \(\{0.5,1,2\}\);
- \(K\in\{1,2\}\);
- all six additive kernels.

We keep only observed-prefix pairs for which **at least one document is globally unseen**, matching the open-tail certificate's assumption that a previously unseen competitor remains possible.

For each state, every admissible ordered tail subset is enumerated. A "true" certificate requires strict score separation, not merely deterministic tie-breaking.


In [5]:

def ordered_subsets(items):
    items = tuple(items)
    out = [()]
    for r in range(1,len(items)+1):
        out.extend(permutations(items,r))
    return out

def generic_bounds(prefixes,weights,kernel,universe,cap):
    observed = set().union(*(set(p) for p in prefixes.values()))
    LB = {d:0.0 for d in observed}
    UB = {d:0.0 for d in observed}
    unseen_bound = 0.0
    for s,pref in prefixes.items():
        w = float(weights[s])
        L = len(pref)
        pos = {d:r for r,d in enumerate(pref,start=1)}
        tail = w*kernel_value(kernel,L+1,cap)
        unseen_bound += tail
        for d in observed:
            if d in pos:
                v = w*kernel_value(kernel,pos[d],cap)
                LB[d] += v
                UB[d] += v
            else:
                UB[d] += tail
    return LB,UB,unseen_bound,observed

def bound_certificate(prefixes,weights,kernel,universe,K,cap):
    LB,UB,U,observed = generic_bounds(prefixes,weights,kernel,universe,cap)
    if len(observed) < K:
        return False,False,()
    ordered = sorted(observed,key=lambda d:(-LB[d],d))
    top = ordered[:K]
    top_set = set(top)
    outsiders = [d for d in observed if d not in top_set]
    max_out = max([UB[d] for d in outsiders] + [U])
    set_cert = min(LB[d] for d in top) > max_out

    order_cert = bool(set_cert)
    if order_cert:
        for j,d in enumerate(top[:-1]):
            later = top[j+1:] + outsiders
            mx = max([UB[x] for x in later] + [U])
            if not (LB[d] > mx):
                order_cert = False
                break
    return bool(set_cert),bool(order_cert),tuple(top)

def fuse_complete(rankings,weights,kernel,universe,cap):
    scores = {d:0.0 for d in universe}
    for s,seq in rankings.items():
        w = float(weights[s])
        for r,d in enumerate(seq,start=1):
            scores[d] += w*kernel_value(kernel,r,cap)
    ordered = sorted(universe,key=lambda d:(-scores[d],d))
    return ordered,scores

def brute_strict_stability(prefixes,weights,kernel,universe,K,cap,completion_cache):
    sources = list(prefixes)
    per_source = [completion_cache[tuple(prefixes[s])] for s in sources]

    reference_set = None
    reference_order = None
    all_same_set = True
    all_same_order = True
    strict_set_all = True
    strict_order_all = True
    ncomp = 0

    for combo in product(*per_source):
        rankings = {s:list(seq) for s,seq in zip(sources,combo)}
        order,scores = fuse_complete(rankings,weights,kernel,universe,cap)
        top = tuple(order[:K])
        top_set = frozenset(top)

        if reference_set is None:
            reference_set = top_set
            reference_order = top
        else:
            if top_set != reference_set:
                all_same_set = False
            if top != reference_order:
                all_same_order = False

        if K < len(order):
            if not (scores[order[K-1]] > scores[order[K]] + 1e-14):
                strict_set_all = False

        # Exact top-K order requires the top-K set itself to be strictly separated,
        # plus strict adjacent order among top-K members.
        for j in range(K-1):
            if not (scores[order[j]] > scores[order[j+1]] + 1e-14):
                strict_order_all = False

        ncomp += 1

    true_set = bool(all_same_set and strict_set_all)
    true_order = bool(true_set and all_same_order and strict_order_all)
    return true_set,true_order,ncomp

UNIVERSE = ('d0','d1','d2','d3')
CAP = len(UNIVERSE)
PREFIXES = []
for L in [1,2]:
    PREFIXES.extend(permutations(UNIVERSE,L))

# Cache all completions of each possible prefix.
completion_cache = {}
for pref in PREFIXES:
    rem = [d for d in UNIVERSE if d not in pref]
    completion_cache[tuple(pref)] = [tuple(pref)+tail for tail in ordered_subsets(rem)]

WEIGHT_GRID = [0.5,1.0,2.0]
K_GRID = [1,2]

rows = []
counterexamples = []
case_id = 0
completion_total = 0

for p0 in PREFIXES:
    for p1 in PREFIXES:
        observed = set(p0)|set(p1)
        if len(observed) == len(UNIVERSE):
            # Open-tail model requires at least one globally unseen candidate.
            continue
        prefixes = {'s0':list(p0),'s1':list(p1)}
        for w0 in WEIGHT_GRID:
            for w1 in WEIGHT_GRID:
                weights = {'s0':w0,'s1':w1}
                for K in K_GRID:
                    for kernel in KERNELS:
                        cert_set,cert_order,cert_top = bound_certificate(
                            prefixes,weights,kernel,UNIVERSE,K,CAP
                        )
                        true_set,true_order,ncomp = brute_strict_stability(
                            prefixes,weights,kernel,UNIVERSE,K,CAP,completion_cache
                        )
                        completion_total += ncomp
                        row = {
                            'case':case_id,
                            'kernel':kernel,
                            'K':K,
                            'w0':w0,'w1':w1,
                            'prefix0':' '.join(p0),
                            'prefix1':' '.join(p1),
                            'n_completions':ncomp,
                            'certificate_set':cert_set,
                            'true_set_stable':true_set,
                            'certificate_order':cert_order,
                            'true_order_stable':true_order,
                        }
                        rows.append(row)
                        if cert_set != true_set or cert_order != true_order:
                            counterexamples.append({
                                **row,
                                'certificate_top':' '.join(cert_top),
                            })
                        case_id += 1

audit_df = pd.DataFrame(rows)
counter_df = pd.DataFrame(counterexamples)
audit_df.to_csv(OUT/'tables'/'systematic_certificate_audit.csv',index=False)
counter_df.to_csv(OUT/'tables'/'systematic_certificate_counterexamples.csv',index=False)

summary_rows = []
for kernel,g in audit_df.groupby('kernel'):
    summary_rows.append({
        'kernel':kernel,
        'cases':len(g),
        'completion_evaluations':int(g.n_completions.sum()),
        'set_false_positive':int(((g.certificate_set==1)&(g.true_set_stable==0)).sum()),
        'set_false_negative':int(((g.certificate_set==0)&(g.true_set_stable==1)).sum()),
        'order_false_positive':int(((g.certificate_order==1)&(g.true_order_stable==0)).sum()),
        'order_false_negative':int(((g.certificate_order==0)&(g.true_order_stable==1)).sum()),
    })
cert_summary = pd.DataFrame(summary_rows)
cert_summary.to_csv(OUT/'tables'/'systematic_certificate_summary.csv',index=False)

display(cert_summary)
print('Systematic prefix states:', len(audit_df))
print('Completion evaluations:', completion_total)
print('Counterexamples:', len(counter_df))
if len(counter_df):
    display(counter_df.head(20))


,kernel,cases,completion_evaluations,set_false_positive,set_false_negative,order_false_positive,order_false_negative
0,borda,4176,265968,0,0,0,0
1,exp20,4176,265968,0,0,0,0
2,inverse_rank,4176,265968,0,0,0,0
3,inverse_sqrt,4176,265968,0,0,0,0
4,log_discount,4176,265968,0,0,0,0
5,rrf60,4176,265968,0,0,0,0


Systematic prefix states: 25056
Completion evaluations: 1595808
Counterexamples: 0


## 5. Decision report and immutable result package

In [ ]:

nested_fail = nested_summary.groupby(['depth','kernel']).agg(
    conditions=('dataset','count'),
    nested_any_order_failure=('nested_order',lambda x:int((x<1.0).sum())),
    nested_any_set_failure=('nested_set',lambda x:int((x<1.0).sum())),
).reset_index()

fp = int(((audit_df.certificate_set==1)&(audit_df.true_set_stable==0)).sum()
         + ((audit_df.certificate_order==1)&(audit_df.true_order_stable==0)).sum())
fn = int(((audit_df.certificate_set==0)&(audit_df.true_set_stable==1)).sum()
         + ((audit_df.certificate_order==0)&(audit_df.true_order_stable==1)).sum())

lines = [
    '# Information Fusion final validation — decision report',
    '',
    '## Frozen-input integrity',
    f'- Canonical manifest present: {manifest_status["present"]}',
    f'- Canonical manifest files checked: {manifest_status["checked"]}',
    f'- Manifest failures: {len(manifest_status["failures"])}',
    f'- SPLADE closure available: {SELF_RUNS is not None}',
    '',
    '## Nested replication audit',
    f'- Real family scenarios: {len(REAL_SCENARIOS)}',
    f'- Depths: {DEPTHS}',
    f'- Kernels: {KERNELS}',
    f'- Added-copy counts: {COPY_EXTRAS}',
    f'- Summary conditions: {len(nested_summary)}',
    '- Family-budget order/set preservation is asserted to be 1.0 in every condition.',
]
for r in nested_fail.itertuples():
    lines.append(
        f'- depth={r.depth}, kernel={r.kernel}: '
        f'nested conditions with any order failure={r.nested_any_order_failure}/{r.conditions}; '
        f'any set failure={r.nested_any_set_failure}/{r.conditions}'
    )

lines += [
    '',
    '## Systematic Stable-style certificate audit',
    f'- Deterministic prefix/weight/kernel/K states evaluated: {len(audit_df)}',
    f'- Total admissible completion evaluations: {completion_total}',
    f'- Total false-positive disagreements (set + order): {fp}',
    f'- Total false-negative disagreements (set + order): {fn}',
    f'- Saved counterexamples: {len(counter_df)}',
    '',
    '## Interpretation',
    '- Nested fusion may be a strong empirical mitigation while still lacking exact replication invariance.',
    '- Family-budget fusion should be claimed for its formal representation guarantee, not universal relevance superiority.',
    '- Zero systematic certificate disagreements are finite verification evidence, not a substitute for a proof.',
]
report = '\n'.join(lines)
(OUT/'DECISION_REPORT.md').write_text(report,encoding='utf-8')
print(report)

protocol = {
    'purpose':'Information Fusion final validation',
    'depths':DEPTHS,
    'top_k':TOP_K,
    'copy_extras':COPY_EXTRAS,
    'kernels':KERNELS,
    'systematic_universe_size':len(UNIVERSE),
    'systematic_weight_grid':WEIGHT_GRID,
    'systematic_k_grid':K_GRID,
    'systematic_requires_globally_unseen_doc':True,
}
(OUT/'PROTOCOL.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')

manifest = {}
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name != 'OUTPUT_SHA256.json':
        manifest[str(p.relative_to(OUT))] = sha256_file(p)
(OUT/'OUTPUT_SHA256.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

zip_path = ROOT/'InvariantRRF_InformationFusion_Final_Validation_Results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            zf.write(p,arcname=str(p.relative_to(OUT)))

print(zip_path)
print('SHA256:', sha256_file(zip_path))
